<a href="https://colab.research.google.com/github/techasit239/Final-Project---DADS6003/blob/main/Complexity-Stylistic-Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# CONFIG
# =========================
INPUT_PATH = "train.xlsx"            # ← ใส่พาธไฟล์ (.xlsx หรือ .csv)
IS_EXCEL   = True                   # ← ถ้าเป็น CSV ให้ตั้งเป็น False
OUTPUT_CSV = "email_features.csv"   # ← ชื่อไฟล์เอาต์พุต

# =========================
# IMPORTS
# =========================
import re
from statistics import pstdev
from typing import List, Tuple
from pathlib import Path
import pandas as pd

# =========================
# LOADERS
# =========================
def load_email_excel(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"ไม่พบไฟล์: {p.resolve()}")
    df = pd.read_excel(p, engine="openpyxl")
    _validate_cols(df)
    _sanitize_cols(df)
    return df

def load_email_csv(path: str, encoding: str = "utf-8") -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"ไม่พบไฟล์: {p.resolve()}")
    df = pd.read_csv(p, encoding=encoding)
    _validate_cols(df)
    _sanitize_cols(df)
    return df

def _validate_cols(df: pd.DataFrame):
    required = {"Subject", "Body", "Label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"คอลัมน์หายไป: {missing} (ต้องมี {required})")

def _sanitize_cols(df: pd.DataFrame):
    df["Subject"] = df["Subject"].astype(str).fillna("")
    df["Body"]    = df["Body"].astype(str).fillna("")
    # ตามต้องการ: แปลง Label เป็น int ก็ได้ (คอมเมนต์บรรทัดถัดไปถ้า label เป็น string)
    # df["Label"] = pd.to_numeric(df["Label"], errors="coerce").fillna(-1).astype(int)

# =========================
# FEATURE EXTRACTORS
# (ครบตามรูป: Complexity + Stylistic)
# =========================
POLITENESS = ["please", "thank", "appreciate", "thanks", "appreciated", "appreciates", "appreciation"]
AGGRESSIVE = ["must", "now", "immediately"]
URGENCY    = ["urgent", "asap", "immediately"]
CONDITIONAL = ["if", "unless"]
PERSONAL_TAGS = ["[recipient’s name]", "[recipient's name],[Your Name],[Your name]"]   # รองรับ ’ และ '
WORD_RE = re.compile(r"[a-zA-Z]+(?:'[a-zA-Z]+)?")

def strip_urls_emails(text: str) -> str:
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)     # ตัด URL
    text = re.sub(r"\S+@\S+\.\S+", " ", text)              # ตัดอีเมล
    return text

def tokenize_words(text: str) -> List[str]:
    text = text if isinstance(text, str) else ""
    text = strip_urls_emails(text)
    return [m.group(0).lower() for m in WORD_RE.finditer(text)]

def make_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def count_markers(tokens: List[str], vocab: List[str]) -> int:
    vocab_set = set(w.lower() for w in vocab)
    return sum(1 for t in tokens if t in vocab_set)

def count_phrase_occurrences(text: str, phrases: List[str]) -> int:
    lowered = (text or "").lower()
    return sum(len(re.findall(re.escape(p.lower()), lowered)) for p in phrases)

def extract_stylo_features(email_text: str) -> dict:
    tokens = tokenize_words(email_text)

    # Complexity
    bigrams  = make_ngrams(tokens, 2)
    trigrams = make_ngrams(tokens, 3)
    word_lengths = [len(w) for w in tokens] or [0]
    word_len_var = pstdev(word_lengths)  # ส่วนเบี่ยงเบนมาตรฐาน (population)

    # Stylistic
    politeness_cnt   = count_markers(tokens, POLITENESS)
    aggressive_cnt   = count_markers(tokens, AGGRESSIVE)
    urgency_cnt      = count_markers(tokens, URGENCY)
    conditional_cnt  = count_markers(tokens, CONDITIONAL)
    personal_token_cnt = count_markers(tokens, ["you", "your"])
    personal_tag_cnt   = count_phrase_occurrences(email_text, PERSONAL_TAGS)
    personalisation_cnt = personal_token_cnt + personal_tag_cnt

    return {
        # Complexity
        "bigram_total_count": len(bigrams),
        "bigram_unique_count": len(set(bigrams)),
        "trigram_total_count": len(trigrams),
        "trigram_unique_count": len(set(trigrams)),
        "word_length_variation_std": float(word_len_var),
        # Stylistic
        "politeness_markers_count": politeness_cnt,
        "aggressiveness_markers_count": aggressive_cnt,
        "urgency_markers_count": urgency_cnt,
        "conditional_phrases_count": conditional_cnt,
        "personalisation_markers_count": personalisation_cnt,
    }

# =========================
# MAIN: read → combine → featurize → save
# =========================
if __name__ == "__main__":
    df = load_email_excel(INPUT_PATH) if IS_EXCEL else load_email_csv(INPUT_PATH)

    # รวม Subject + Body (จะโฟกัสข้อความมากกว่า URL/อีเมล เพราะเราตัดทิ้งแล้ว)
    combined = df["Subject"].fillna("") + "\n" + df["Body"].fillna("")

    # ดึงฟีเจอร์ทีละแถวแล้วแปลงเป็น DataFrame
    features_df = combined.apply(extract_stylo_features).apply(pd.Series)

    # ต่อคอลัมน์เดิม + ฟีเจอร์ แล้วเซฟ
    out = pd.concat([df.reset_index(drop=True), features_df], axis=1)
    out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    # แสดงสรุป
    print(f"บันทึกไฟล์แล้ว → {OUTPUT_CSV}")
    print(out.head())
